In [0]:
# 1. Install pyyaml and openpyxl dependency for yaml and xml file format usage
%pip install pyyaml openpyxl


In [0]:
dbutils.library.restartPython()

In [0]:

# 2. Import libraries
import yaml
import pandas as pd

In [0]:
# 3. Defined common read functions as per the file formats

def read_csv(spark, config):
    return (
        spark.read.format("csv")
        .options(**config.get("options", {}))
        .load(config["source_path"])
    )


def read_json(spark, config):
    return (
        spark.read.format("json")
        .options(**config.get("options", {}))
        .load(config["source_path"])
    )

# SPark can't read xlsx files directly, so we need to use pandas to read the file
def read_xlsx(spark, config):
    import pandas as pd

    df_pd = pd.read_excel(config["source_path"])

    # Fix phone columns: Phone number contains special symbols, hence it is read as string
    if "phone" in df_pd.columns:
        df_pd["phone"] = df_pd["phone"].astype(str)

    return spark.createDataFrame(df_pd)

In [0]:
# Defined readers to read different file formats
READERS = {
    "csv": read_csv,
    "json": read_json,
    "xlsx": read_xlsx
}

# This function takes input as the and returns which reader function to use to read the dataset based on the file format
def get_reader(format):
    if format not in READERS:
        raise ValueError(f"Unsupported format: {format}")
    return READERS[format]

In [0]:
# orders_ingest.yaml file contains details of all the dataset which needs to be ingested in bronze layer
config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/bronze/orders_ingest.yaml"

# Reading the configuration file file
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Adding audit columns: ingestion_timestamp and source_file to capture the details of the dataset
def standardize(df, config):
    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(config["source_path"]))
    )

In [0]:
def handle_pii(df, config):

    pii_cols = config.get("pii_columns", [])
    strategy = config.get("pii_strategy", "none")

    if strategy == "hash":
        from pyspark.sql.functions import sha2, col
        for c in pii_cols:
            df = df.withColumn(f"{c}_hash", sha2(col(c), 256))
        df = df.drop(*pii_cols)

    elif strategy == "drop":
        df = df.drop(*pii_cols)

    return df

In [0]:
def write_to_bronze(df, config):
    target_path = config["target_path"]

# Loading data in Bronze layer with no changes but adding the audit Columns
    (
        df.write
        .mode(config.get("write_mode", "overwrite"))
        .format("parquet") 
        .save(target_path)
    )

In [0]:
def ingest_dataset(config):

    #To get which function needs to be executed based on the file format to be read from raw
    reader_fn = get_reader(config["format"])
    df = reader_fn(spark, config)
    df = standardize(df, config)
    df = handle_pii(df, config)
    write_to_bronze(df, config)

In [0]:
for dataset in config["datasets"]:
    try:
        ingest_dataset(dataset)
    except Exception as e:
        print(f"Failed: {dataset['name']} → {str(e)}")

In [0]:
orders_df = spark.read.parquet("/Volumes/pei/bronze/orders/")
customers_df = spark.read.parquet("/Volumes/pei/bronze/customer/")
products_df = spark.read.parquet("/Volumes/pei/bronze/product/")